# Unidad 03
## Pivoteando

In [ ]:
USE Pampero;
DROP TABLE IF EXISTS dbo.Ordenes;
CREATE TABLE dbo.Ordenes
(
ordenid INT NOT NULL,
fechaorden DATE NOT NULL,
empid INT NOT NULL,
clienteid VARCHAR(5) NOT NULL,
cantidad INT NOT NULL,
CONSTRAINT PK_Orders PRIMARY KEY(ordenid)
);

Commands completed successfully.

Total execution time: 00:00:00.023

In [ ]:
INSERT INTO dbo.Ordenes(ordenid, fechaorden, empid, clienteid, cantidad) VALUES
(30001, '20140802', 3, 'A', 10),
(10001, '20141224', 2, 'A', 12),
(10005, '20141224', 1, 'B', 20),
(40001, '20150109', 2, 'A', 40),
(10006, '20150118', 1, 'C', 14),
(20001, '20150212', 2, 'B', 12),
(40005, '20160212', 3, 'A', 10),
(20002, '20160216', 1, 'C', 20),
(30003, '20160418', 2, 'B', 15),
(30004, '20140418', 3, 'C', 22),
(30007, '20160907', 3, 'D', 30);
SELECT * FROM dbo.Ordenes;

(11 rows affected)
(11 rows affected)

ordenid | fechaorden | empid | clienteid | cantidad
--------+------------+-------+-----------+---------
10001   | 2014-12-24 | 2     | A         | 12      
10005   | 2014-12-24 | 1     | B         | 20      
10006   | 2015-01-18 | 1     | C         | 14      
20001   | 2015-02-12 | 2     | B         | 12      
20002   | 2016-02-16 | 1     | C         | 20      
30001   | 2014-08-02 | 3     | A         | 10      
30003   | 2016-04-18 | 2     | B         | 15      
30004   | 2014-04-18 | 3     | C         | 22      
30007   | 2016-09-07 | 3     | D         | 30      
40001   | 2015-01-09 | 2     | A         | 40      
40005   | 2016-02-12 | 3     | A         | 10      
(11 rows)

Total execution time: 00:00:00.114

In [ ]:
SELECT empid, clienteid, SUM(cantidad) AS sumqty
FROM dbo.Ordenes
GROUP BY empid, clienteid;


(7 rows affected)

empid | clienteid | sumqty
------+-----------+-------
2     | A         | 52    
3     | A         | 20    
1     | B         | 20    
2     | B         | 27    
1     | C         | 34    
3     | C         | 22    
3     | D         | 30    
(7 rows)

Total execution time: 00:00:00.043

Pero lo necesito así:
|empid|A|B|C|D|
|-----|-|-|-|-|
|1|NULL|20|34|NULL|
|2|52|27|NULL|NULL|
|3|20|NULL|22|30|

In [ ]:
SELECT empid,
SUM(CASE WHEN clienteid = 'A' THEN cantidad END) AS A,
SUM(CASE WHEN clienteid = 'B' THEN cantidad END) AS B,
SUM(CASE WHEN clienteid = 'C' THEN cantidad END) AS C,
SUM(CASE WHEN clienteid = 'D' THEN cantidad END) AS D
FROM dbo.Ordenes
GROUP BY empid;

(3 rows affected)

empid | A    | B    | C    | D   
------+------+------+------+-----
1     | NULL | 20   | 34   | NULL
2     | 52   | 27   | NULL | NULL
3     | 20   | NULL | 22   | 30  
(3 rows)

Total execution time: 00:00:00.019

In [ ]:
SELECT empid, A, B, C, D
FROM (SELECT empid, clienteid, cantidad
FROM dbo.Ordenes) AS D
PIVOT(SUM(cantidad) FOR clienteid IN(A, B, C, D)) AS P;

(3 rows affected)

empid | A    | B    | C    | D   
------+------+------+------+-----
1     | NULL | 20   | 34   | NULL
2     | 52   | 27   | NULL | NULL
3     | 20   | NULL | 22   | 30  
(3 rows)

Total execution time: 00:00:00.029

In [ ]:
SELECT clienteid, [1], [2], [3]
FROM (SELECT empid, clienteid, cantidad
FROM dbo.Ordenes) AS D
PIVOT(SUM(cantidad) FOR empid IN([1], [2], [3])) AS P;

(4 rows affected)

clienteid | 1    | 2    | 3   
----------+------+------+-----
A         | NULL | 52   | 20  
B         | 20   | 27   | NULL
C         | 34   | NULL | 22  
D         | NULL | NULL | 30  
(4 rows)

Total execution time: 00:00:00.025

In [ ]:
DECLARE @colnameList varchar(200)
SET @colnameList = NULL
SELECT @colnameList = COALESCE(@colnameList + ',','') + clienteid
FROM (SELECT DISTINCT clienteid FROM dbo.Ordenes) AS O;
DECLARE @SQLQuery NVARCHAR(MAX);
SET @SQLQuery =
'SELECT empid, ' + @colnameList + '
FROM (SELECT empid, clienteid, cantidad
FROM dbo.Ordenes) AS D
PIVOT(SUM(cantidad) FOR clienteid IN(' + @colnameList + ')) AS P'
EXEC(@SQLQuery)

(3 rows affected)

empid | A    | B    | C    | D   
------+------+------+------+-----
1     | NULL | 20   | 34   | NULL
2     | 52   | 27   | NULL | NULL
3     | 20   | NULL | 22   | 30  
(3 rows)

Total execution time: 00:00:00.020

## Despivoteando

In [9]:
USE Pampero;
DROP TABLE IF EXISTS dbo.EmpClieOrdenes;
CREATE TABLE dbo.EmpClieOrdenes
(
empid INT NOT NULL
CONSTRAINT PK_EmpClieOrdenes PRIMARY KEY,
A VARCHAR(5) NULL,
B VARCHAR(5) NULL,
C VARCHAR(5) NULL,
D VARCHAR(5) NULL
);

Commands completed successfully.

Total execution time: 00:00:00.050

In [10]:
INSERT INTO dbo.EmpClieOrdenes(empid, A, B, C, D)
SELECT empid, A, B, C, D
FROM (SELECT empid, clienteid, cantidad
FROM dbo.Ordenes) AS D
PIVOT(SUM(cantidad) FOR clienteid IN(A, B, C, D)) AS P;
SELECT * FROM dbo.EmpClieOrdenes;

(3 rows affected)
(3 rows affected)

empid | A    | B    | C    | D   
------+------+------+------+-----
1     | NULL | 20   | 34   | NULL
2     | 52   | 27   | NULL | NULL
3     | 20   | NULL | 22   | 30  
(3 rows)

Total execution time: 00:00:00.033

In [11]:
SELECT *
FROM dbo.EmpClieOrdenes
CROSS JOIN (VALUES('A'),('B'),('C'),('D')) AS C(clienteid);

(12 rows affected)

empid | A    | B    | C    | D    | clienteid
------+------+------+------+------+----------
1     | NULL | 20   | 34   | NULL | A        
1     | NULL | 20   | 34   | NULL | B        
1     | NULL | 20   | 34   | NULL | C        
1     | NULL | 20   | 34   | NULL | D        
2     | 52   | 27   | NULL | NULL | A        
2     | 52   | 27   | NULL | NULL | B        
2     | 52   | 27   | NULL | NULL | C        
2     | 52   | 27   | NULL | NULL | D        
3     | 20   | NULL | 22   | 30   | A        
3     | 20   | NULL | 22   | 30   | B        
3     | 20   | NULL | 22   | 30   | C        
3     | 20   | NULL | 22   | 30   | D        
(12 rows)

Total execution time: 00:00:00.019

In [12]:
SELECT empid, clienteid, cantidad
FROM dbo.EmpClieOrdenes
CROSS APPLY (VALUES('A', A),('B', B),('C', C),('D', D)) AS C(clienteid, cantidad);

(12 rows affected)

empid | clienteid | cantidad
------+-----------+---------
1     | A         | NULL    
1     | B         | 20      
1     | C         | 34      
1     | D         | NULL    
2     | A         | 52      
2     | B         | 27      
2     | C         | NULL    
2     | D         | NULL    
3     | A         | 20      
3     | B         | NULL    
3     | C         | 22      
3     | D         | 30      
(12 rows)

Total execution time: 00:00:00.066

In [13]:
SELECT empid, clienteid, cantidad
FROM dbo.EmpClieOrdenes
CROSS APPLY (VALUES('A', A),('B', B),('C', C),('D', D)) AS C(clienteid, cantidad)
WHERE cantidad IS NOT NULL;

(7 rows affected)

empid | clienteid | cantidad
------+-----------+---------
1     | B         | 20      
1     | C         | 34      
2     | A         | 52      
2     | B         | 27      
3     | A         | 20      
3     | C         | 22      
3     | D         | 30      
(7 rows)

Total execution time: 00:00:00.014

In [14]:
SELECT empid, clienteid, cantidad
FROM dbo.EmpClieOrdenes
    UNPIVOT(cantidad FOR clienteid IN (A, B, C, D)) AS U;

(7 rows affected)

empid | clienteid | cantidad
------+-----------+---------
1     | B         | 20      
1     | C         | 34      
2     | A         | 52      
2     | B         | 27      
3     | A         | 20      
3     | C         | 22      
3     | D         | 30      
(7 rows)

Total execution time: 00:00:00.015

## Conjuntos de agrupación

In [15]:
SELECT empid, clienteid, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY empid, clienteid;
SELECT empid, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY empid;
SELECT clienteid, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY clienteid;
SELECT SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes;

(7 rows affected)
(3 rows affected)
(4 rows affected)
(1 row affected)

empid | clienteid | sumcantidad
------+-----------+------------
2     | A         | 52         
3     | A         | 20         
1     | B         | 20         
2     | B         | 27         
1     | C         | 34         
3     | C         | 22         
3     | D         | 30         
(7 rows)

empid | sumcantidad
------+------------
1     | 54         
2     | 79         
3     | 72         
(3 rows)

clienteid | sumcantidad
----------+------------
A         | 72         
B         | 47         
C         | 56         
D         | 30         
(4 rows)

sumcantidad
-----------
205        
(1 row)

Total execution time: 00:00:00.039

In [16]:
SELECT empid, clienteid, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY empid, clienteid
UNION ALL
SELECT empid, NULL, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY empid
UNION ALL
SELECT NULL, clienteid, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY clienteid
UNION ALL
SELECT NULL, NULL, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes;

(15 rows affected)

empid | clienteid | sumcantidad
------+-----------+------------
2     | A         | 52         
3     | A         | 20         
1     | B         | 20         
2     | B         | 27         
1     | C         | 34         
3     | C         | 22         
3     | D         | 30         
1     | NULL      | 54         
2     | NULL      | 79         
3     | NULL      | 72         
NULL  | A         | 72         
NULL  | B         | 47         
NULL  | C         | 56         
NULL  | D         | 30         
NULL  | NULL      | 205        
(15 rows)

Total execution time: 00:00:00.027

In [17]:
SELECT empid, clienteid, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY
GROUPING SETS
(
(empid, clienteid),
(empid),
(clienteid),
()
);

(15 rows affected)

empid | clienteid | sumcantidad
------+-----------+------------
2     | A         | 52         
3     | A         | 20         
NULL  | A         | 72         
1     | B         | 20         
2     | B         | 27         
NULL  | B         | 47         
1     | C         | 34         
3     | C         | 22         
NULL  | C         | 56         
3     | D         | 30         
NULL  | D         | 30         
NULL  | NULL      | 205        
1     | NULL      | 54         
2     | NULL      | 79         
3     | NULL      | 72         
(15 rows)

Total execution time: 00:00:00.032

In [18]:
SELECT empid, clienteid, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY CUBE(empid, clienteid);

(15 rows affected)

empid | clienteid | sumcantidad
------+-----------+------------
2     | A         | 52         
3     | A         | 20         
NULL  | A         | 72         
1     | B         | 20         
2     | B         | 27         
NULL  | B         | 47         
1     | C         | 34         
3     | C         | 22         
NULL  | C         | 56         
3     | D         | 30         
NULL  | D         | 30         
NULL  | NULL      | 205        
1     | NULL      | 54         
2     | NULL      | 79         
3     | NULL      | 72         
(15 rows)

Total execution time: 00:00:00.038

In [19]:
SELECT
YEAR(fechaorden) AS anioorden,
MONTH(fechaorden) AS mesorden,
DAY(fechaorden) AS diaorden,
SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY ROLLUP(YEAR(fechaorden), MONTH(fechaorden), DAY(fechaorden));

(22 rows affected)

anioorden | mesorden | diaorden | sumcantidad
----------+----------+----------+------------
2014      | 4        | 18       | 22         
2014      | 4        | NULL     | 22         
2014      | 8        | 2        | 10         
2014      | 8        | NULL     | 10         
2014      | 12       | 24       | 32         
2014      | 12       | NULL     | 32         
2014      | NULL     | NULL     | 64         
2015      | 1        | 9        | 40         
2015      | 1        | 18       | 14         
2015      | 1        | NULL     | 54         
2015      | 2        | 12       | 12         
2015      | 2        | NULL     | 12         
2015      | NULL     | NULL     | 66         
2016      | 2        | 12       | 10         
2016      | 2        | 16       | 20         
2016      | 2        | NULL     | 30         
2016      | 4        | 18       | 15         
2016      | 4        | NULL     | 15         
2016      | 9        | 7        | 30         
2016      | 9 

In [20]:
SELECT
GROUPING(empid) AS grpemp,
GROUPING(clienteid) AS grpcli,
empid, clienteid, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY CUBE(empid, clienteid);

(15 rows affected)

grpemp | grpcli | empid | clienteid | sumcantidad
-------+--------+-------+-----------+------------
0      | 0      | 2     | A         | 52         
0      | 0      | 3     | A         | 20         
1      | 0      | NULL  | A         | 72         
0      | 0      | 1     | B         | 20         
0      | 0      | 2     | B         | 27         
1      | 0      | NULL  | B         | 47         
0      | 0      | 1     | C         | 34         
0      | 0      | 3     | C         | 22         
1      | 0      | NULL  | C         | 56         
0      | 0      | 3     | D         | 30         
1      | 0      | NULL  | D         | 30         
1      | 1      | NULL  | NULL      | 205        
0      | 1      | 1     | NULL      | 54         
0      | 1      | 2     | NULL      | 79         
0      | 1      | 3     | NULL      | 72         
(15 rows)

Total execution time: 00:00:00.030

In [21]:
SELECT
GROUPING_ID(empid, clienteid) AS groupingset,
empid, clienteid, SUM(cantidad) AS sumcantidad
FROM dbo.Ordenes
GROUP BY CUBE(empid, clienteid);

(15 rows affected)

groupingset | empid | clienteid | sumcantidad
------------+-------+-----------+------------
0           | 2     | A         | 52         
0           | 3     | A         | 20         
2           | NULL  | A         | 72         
0           | 1     | B         | 20         
0           | 2     | B         | 27         
2           | NULL  | B         | 47         
0           | 1     | C         | 34         
0           | 3     | C         | 22         
2           | NULL  | C         | 56         
0           | 3     | D         | 30         
2           | NULL  | D         | 30         
3           | NULL  | NULL      | 205        
1           | 1     | NULL      | 54         
1           | 2     | NULL      | 79         
1           | 3     | NULL      | 72         
(15 rows)

Total execution time: 00:00:00.024